In [2]:
import csv

# File paths
input_file_path = 'Vehicles.csv'
output_file_path = 'Dim_vehicle.csv'

# Load the CSV file into a list of dictionaries
vehicles_data = []
try:
    with open(input_file_path, 'r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            # Ensure numeric fields are handled properly
            row["VEHICLE_ID"] = row["VEHICLE_ID"] if row["VEHICLE_ID"] else None
            row["VEHICLE_YEAR"] = int(row["VEHICLE_YEAR"]) if row["VEHICLE_YEAR"] and row["VEHICLE_YEAR"].isdigit() else None
            row["OCCUPANT_CNT"] = float(row["OCCUPANT_CNT"]) if row["OCCUPANT_CNT"] and row["OCCUPANT_CNT"].replace('.', '', 1).isdigit() else None
            vehicles_data.append(row)
except FileNotFoundError:
    print(f"Error: File {input_file_path} not found.")
    vehicles_data = []

# Define vehicle and non-vehicle types
vehicle_types = ["DRIVER", "PARKED", "DRIVERLESS"]
non_vehicle_types = ["PEDESTRIAN", "BICYCLE", "NON-MOTOR VEHICLE", "NON-CONTACT VEHICLE"]

# Initialize counters
missing_counts = {
    "UNIT_TYPE": 0,
    "VEHICLE_ID": 0,
    "VEHICLE_YEAR": 0,
    "OCCUPANT_CNT": 0
}

# Process the data to handle missing values
for row in vehicles_data:
    # Handle UNIT_TYPE
    if not row["UNIT_TYPE"] or row["UNIT_TYPE"].strip() == "":
        row["UNIT_TYPE"] = "Unknown"
        missing_counts["UNIT_TYPE"] += 1

    # Handle VEHICLE_ID
    if row["VEHICLE_ID"] is None:
        if row["UNIT_TYPE"] in vehicle_types:
            row["VEHICLE_ID"] = f"{row['RD_NO']}_{row['UNIT_NO']}"  # Synthetic ID
        else:
            row["VEHICLE_ID"] = -1  # Assign -1 for non-vehicles
        missing_counts["VEHICLE_ID"] += 1

    # Handle VEHICLE_YEAR with -1 for missing values
    if row["VEHICLE_YEAR"] is None:
        row["VEHICLE_YEAR"] = -1
        missing_counts["VEHICLE_YEAR"] += 1

    # Handle OCCUPANT_CNT with -1 for missing values
    if row["OCCUPANT_CNT"] is None:
        row["OCCUPANT_CNT"] = -1
        missing_counts["OCCUPANT_CNT"] += 1

    # Handle MODEL, MAKE, LIC_PLATE_STATE, and other fields
    for key in ["MODEL", "MAKE", "LIC_PLATE_STATE", "VEHICLE_DEFECT", "VEHICLE_TYPE", "VEHICLE_USE", "TRAVEL_DIRECTION", "MANEUVER", "FIRST_CONTACT_POINT"]:
        if not row[key] or row[key].strip() == "":
            row[key] = "Unknown"

# Save the cleaned data to a new CSV file
try:
    with open(output_file_path, 'w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=vehicles_data[0].keys())
        writer.writeheader()
        writer.writerows(vehicles_data)
    print(f"Cleaned data saved to {output_file_path}.")
except Exception as e:
    print(f"An error occurred while saving the file: {e}")

# Display the counts of fixed missing values
print("Counts of Fixed Missing Values:")
for column, count in missing_counts.items():
    print(f"{column}: {count}")

Cleaned data saved to Dim_vehicle.csv.
Counts of Fixed Missing Values:
UNIT_TYPE: 1
VEHICLE_ID: 10373
VEHICLE_YEAR: 460437
OCCUPANT_CNT: 10373
